# Silver — Transformacion y Calidad de Datos
**LogiLake | D'LOGIA** — Capa Silver del Medallion Architecture

Operaciones en esta capa:
1. Casteo de tipos (strings -> timestamps, floats)
2. Columnas derivadas de logistica (dias de entrega, retraso, OTIF flag)
3. Validaciones de calidad de datos (DQ flags)
4. Deduplicacion por `order_id`
5. Escritura en Delta Lake Silver (UPSERT)

In [ ]:
# ── SparkSession con Delta Lake 3.1.0 (PySpark 3.5.0) ────────────────────────
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("logilake_silver")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Delta Lake activo")

In [ ]:
# ── Rutas locales ─────────────────────────────────────────────────────────────
import os
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BASE_PATH   = "./data"
BRONZE_PATH = f"{BASE_PATH}/bronze/olist_orders"
SILVER_PATH = f"{BASE_PATH}/silver/olist_orders"

os.makedirs(SILVER_PATH, exist_ok=True)
print(f"Bronze: {BRONZE_PATH}")
print(f"Silver: {SILVER_PATH}")

In [ ]:
# ── 1. Leer Bronze ────────────────────────────────────────────────────────────
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze registros: {bronze_df.count():,}")
bronze_df.show(3, truncate=True)

In [ ]:
# ── 2. Casteo de tipos + columnas derivadas de logistica ──────────────────────
TS_FORMAT = "yyyy-MM-dd HH:mm:ss"

silver_df = (
    bronze_df

    # Timestamps
    .withColumn("order_purchase_ts",
        F.to_timestamp(F.col("order_purchase_ts"), TS_FORMAT))
    .withColumn("order_approved_ts",
        F.to_timestamp(F.col("order_approved_ts"), TS_FORMAT))
    .withColumn("order_delivered_ts",
        F.to_timestamp(F.col("order_delivered_ts"), TS_FORMAT))
    .withColumn("order_estimated_ts",
        F.to_timestamp(F.col("order_estimated_ts"), TS_FORMAT))

    # Dias de entrega reales
    .withColumn("delivery_days_actual",
        F.datediff(
            F.col("order_delivered_ts"),
            F.col("order_purchase_ts")
        ).cast("integer"))

    # Dias de entrega estimados
    .withColumn("delivery_days_estimated",
        F.datediff(
            F.col("order_estimated_ts"),
            F.col("order_purchase_ts")
        ).cast("integer"))

    # Retraso (positivo = tardio, negativo = adelantado)
    .withColumn("delivery_delay_days",
        F.when(
            F.col("order_delivered_ts").isNotNull() & F.col("order_estimated_ts").isNotNull(),
            F.datediff(F.col("order_delivered_ts"), F.col("order_estimated_ts"))
        ).otherwise(F.lit(None)).cast("integer"))

    # Flags de estado
    .withColumn("is_delivered",
        F.col("order_status") == "delivered")
    .withColumn("is_canceled",
        F.col("order_status") == "canceled")
    .withColumn("is_late_delivery",
        F.when(
            F.col("is_delivered") & F.col("delivery_delay_days").isNotNull(),
            F.col("delivery_delay_days") > 0
        ).otherwise(F.lit(False)))

    # Valor total de la orden
    .withColumn("order_value_total",
        F.col("total_items_value").cast("double") +
        F.col("total_freight").cast("double"))

    # Ratio de flete
    .withColumn("freight_ratio",
        F.when(
            F.col("total_items_value").cast("double") > 0,
            F.col("total_freight").cast("double") /
            F.col("total_items_value").cast("double")
        ).otherwise(F.lit(None)))

    # Metadato Silver
    .withColumn("silver_processed_at", F.current_timestamp())
)

print("Columnas Silver:", len(silver_df.columns))
silver_df.printSchema()

In [ ]:
# ── 3. Validaciones de Calidad de Datos (DQ) ──────────────────────────────────
dq_flags_col = F.array(
    F.when(F.col("order_id").isNull(),
           F.lit("NULL_ORDER_ID")).otherwise(F.lit(None)),
    F.when(F.col("payment_value").isNull() | (F.col("payment_value") <= 0),
           F.lit("INVALID_PAYMENT_VALUE")).otherwise(F.lit(None)),
    F.when(F.col("item_count").isNull() | (F.col("item_count") < 1),
           F.lit("INVALID_ITEM_COUNT")).otherwise(F.lit(None)),
    F.when(
        F.col("order_delivered_ts").isNotNull() &
        F.col("order_purchase_ts").isNotNull() &
        (F.col("delivery_days_actual") < 0),
        F.lit("DELIVERY_BEFORE_PURCHASE")
    ).otherwise(F.lit(None)),
    F.when(
        F.col("review_score").isNotNull() &
        ((F.col("review_score") < 1) | (F.col("review_score") > 5)),
        F.lit("INVALID_REVIEW_SCORE")
    ).otherwise(F.lit(None)),
    F.when(
        F.col("delivery_days_actual").isNotNull() & (F.col("delivery_days_actual") > 120),
        F.lit("ANOMALOUS_DELIVERY_DAYS")
    ).otherwise(F.lit(None)),
)

silver_dq = (
    silver_df
    .withColumn("_dq_flags_arr", dq_flags_col)
    .withColumn("dq_flags",
        F.array_join(
            F.filter(F.col("_dq_flags_arr"), lambda x: x.isNotNull()),
            "|"
        )
    )
    .withColumn("dq_passed",
        F.size(F.filter(F.col("_dq_flags_arr"), lambda x: x.isNotNull())) == 0
    )
    .drop("_dq_flags_arr")
)

total  = silver_dq.count()
passed = silver_dq.filter(F.col("dq_passed")).count()
failed = total - passed
print(f"Total:   {total:,}")
print(f"DQ OK:   {passed:,} ({passed/total*100:.1f}%)")
print(f"DQ FAIL: {failed:,} ({failed/total*100:.1f}%)")

print("\nDistribucion de flags:")
silver_dq.filter(~F.col("dq_passed")) \
    .groupBy("dq_flags").count().orderBy(F.desc("count")).show()

In [ ]:
# ── 4. Deduplicacion por order_id ─────────────────────────────────────────────
window_spec = Window.partitionBy("order_id").orderBy(F.desc("bronze_ingested_at"))

silver_dedup = (
    silver_dq
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

print(f"Antes dedup:   {silver_dq.count():,}")
print(f"Despues dedup: {silver_dedup.count():,}")

In [ ]:
# ── 5. Escribir Silver (UPSERT con Delta MERGE) ───────────────────────────────
if not DeltaTable.isDeltaTable(spark, SILVER_PATH):
    silver_dedup.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("order_status") \
        .save(SILVER_PATH)
    print(f"Silver creado: {silver_dedup.count():,} filas")
else:
    silver_table = DeltaTable.forPath(spark, SILVER_PATH)
    (
        silver_table.alias("target")
        .merge(
            silver_dedup.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("MERGE Silver completado")

In [ ]:
# ── 6. Verificacion Silver ────────────────────────────────────────────────────
silver_final = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver total: {silver_final.count():,} filas\n")

print("KPI preview — metricas de logistica:")
silver_final.filter(F.col("is_delivered")).agg(
    F.round(F.avg("delivery_days_actual"), 1).alias("avg_delivery_days"),
    F.round(F.avg("delivery_days_estimated"), 1).alias("avg_estimated_days"),
    F.round(F.avg("delivery_delay_days"), 1).alias("avg_delay_days"),
    F.round(F.mean(F.col("is_late_delivery").cast("integer")) * 100, 1).alias("pct_late"),
    F.round(F.avg("payment_value"), 2).alias("avg_payment_value"),
    F.round(F.avg("review_score"), 2).alias("avg_review_score"),
).show(truncate=False)

print("\nDelta history:")
DeltaTable.forPath(spark, SILVER_PATH) \
    .history(3) \
    .select("version", "timestamp", "operation", "operationParameters") \
    .show(truncate=True)